<pre>
#+TITLE: Project 2: Analyzing the 2008 Financial Crisis
#+AUTHOR: Edgar Huamantla
#+DATE: 2026-03-22
</pre>

In [ ]:
# Because jupyter notebook has a global name space; use this area for imports
import pandas as pd
import numpy as np
import plotly.express as px
import math

# Obtain resource files

In [ ]:
import pathlib
import importlib.util

#check if gdown is installed in current python environment; alert the user to activate their virtual environment to avoid installing to global
if importlib.util.find_spec("gdown") is None:
    print("gdown is not installed. Please activate your virtual environment and install gdown to proceed.")

# dictionary: filename and gdown id
file_names = {'USE3L0712.RSK': '1Y0WE4cqbZfdNEvFuWIDR_d6oDbHHNkly',
              'USE3L0812.RSK': '1Z3vdd5m8uoB4whomq92DQbZHJiKYJc0v'}

curr_working_dir = pathlib.Path.cwd()
print(f"Current working directory: {curr_working_dir}")

for file_name, gdown_id in file_names.items():
    file_path = curr_working_dir / file_name
    if not file_path.exists():
        print(f"File {file_path} does not exist.")
        !gdown {gdown_id} -O {file_path}
    else:
        print(f"File {file_path} already exists. No need to download.")




In [ ]:
# check file integrity of downloaded files
for file_count, file_name in enumerate(file_names.keys(), start=1):
    print(f"\nChecking integrity of file {file_count}: {file_name}...")
    ! head -n 5 {file_name}

# Extract 
 - parse text file into panadas file
 - ignore first row with date ; seen in the head output
    - set second row to header

In [ ]:
barra_df_07 = pd.read_csv('USE3L0712.RSK', skiprows=1, header=0)
print(f"\nDataframe shape: {barra_df_07.shape}")
print("head")
display(barra_df_07.head(3))
print("tail")
display(barra_df_07.tail(3))

In [ ]:
# print DataFrame information showing index dtype and columns, non-NA values and memory usage.
barra_df_07.info()

In [ ]:
# show column names
barra_df_07.columns

In [ ]:
# clean column names; first remove leading and trailing white spaces
barra_df_07.columns = barra_df_07.columns.str.strip()

# check if the column name has invalid characters
invalid_characters = {"%": "_pct"}

# check if column begins with a number; if so, prepend with "col_"
def check_name_leads_with_number(column_name):
    if column_name[0].isdigit():
        print(f"Column name '{column_name}' begins with a number. Prepending with 'col_'.")
        return "col_" + column_name
    else:
        return column_name

# clean column names; replace invalid characters and check if column name begins with a number
for invalid_char, replacement in invalid_characters.items():
    barra_df_07.columns = barra_df_07.columns.str.replace(invalid_char, replacement, regex=False)
barra_df_07.columns = barra_df_07.columns.map(check_name_leads_with_number)

# make lowercase
barra_df_07.columns = barra_df_07.columns.str.lower()


In [ ]:
# new column names
barra_df_07.columns

In [ ]:
# Descriptive statistics include those that summarize the central tendency, dispersion and shape of a dataset’s distribution, excluding NaN values.
barra_df_07.describe()

In [ ]:
# box plot numeric columns; find them first

def find_numeric_columns(df):
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"found {len(numeric_cols)} numeric columns in the DataFrame.")
    print(f"Numeric columns: {numeric_cols}")
    
    return numeric_cols

numeric_columns = find_numeric_columns(barra_df_07)

# to fix the the stack of boxplots we define rows
num_rows = math.ceil(len(numeric_columns) / 4) # 4 columns per row


# melt the df; so that it is not on the same axis
df_melted = barra_df_07.melt(
                            value_vars=numeric_columns,
                            var_name="factor_type", 
                            value_name="factor_value"
                            )


fig = px.box(
    df_melted,
    y="factor_value", # numeric column to plot
    facet_col="factor_type", # categorical column to facet by
    facet_col_wrap=4, # like a line break 
    height = 300 * num_rows, # adjust height based on number of rows needed
    template='plotly_dark',
    points='outliers', # show outliers as points
    facet_col_spacing=0.02, # reduce spacing between facets
    facet_row_spacing=0.02, # reduce spacing between rows of facets
)

fig.update_yaxes(matches=None, showticklabels=True) # allow y axes to have different scales



#clean title
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))


fig.show()

# Transform (Data Cleaning)
 - column names - trim leading and trailing white spaces -- done earlier
    - (optional): set make column names lowercase
 - use a dictionary to replace invalid characters and the value with its replacement (i.e. {% : "_pct"})
 - identify how NaN values are encoded (from box plot hbta has -999)
 - set index

In [ ]:
barra_df_07.head(3)

In [ ]:
# display current indices
barra_df_07.index

In [ ]:
barra_df_07.set_index('ticker').index # by itself does not persist, need to assign back to df or use inplace=True
barra_df_07 = barra_df_07.set_index('ticker') # assign back to df to persist the change

In [ ]:
# find non numbers by column
# numeric_columns already holds numeric columns
non_numeric_columns = [col for col in barra_df_07.columns if col not in numeric_columns]
non_numeric_columns

#optionally: barra_df_07.select_dtypes('object')

In [ ]:
# barra_df_07.loc['AAPL'] error
# barra_df_07[non_numeric_columns].str.strip() # does not work because str returns a series. need to apply to columns separately

def strip_value(value: pd.Series) -> str:
    return value.str.strip()

barra_df_07[non_numeric_columns] = barra_df_07[non_numeric_columns].apply(strip_value)

# lambda way
# barra_df_07[non_numeric_columns] = barra_df_07[non_numeric_columns].apply(lambda x: x.str.strip())

In [ ]:
barra_df_07.loc['AAPL']

## look for null values
- -999 is being used to encode null values
    - search in the numeric columns, return the column

In [ ]:
# intially was going to search all columns, but -999 is numeric so we can just search numeric columns for -999
cols_with_nulls = barra_df_07.columns[(barra_df_07 == -999).any()]
cols_with_nulls

In [ ]:
# null counts
null_counts = (barra_df_07['hbta'] == -999).sum()
null_counts

In [ ]:
# replace the -999 with np.nan
barra_df_07['hbta'] = barra_df_07['hbta'].replace(-999, np.nan)

In [ ]:
# earlier we saw the hbta box plot with the -999 values
fig = px.box(
    barra_df_07['hbta'],
    y='hbta',
    points='outliers',
    notched=True,
    template='plotly_dark',
    title='Box plot of hbta after replacing -999 with NaN'
)
fig.show()